# Phase 11 — Controlled Ablation Architecture Preparation

MorphoNet-BT Architecture v1.0. Preparation and disposable safety diagnostics only; no full ablation training or test evaluation.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


## Scientific objective

Measure the conditional incremental contribution of the multi-scale stem (MSS), privileged morphology supervision, boundary-aware attention (BAA), global/local dual path and adaptive morphological gated fusion (AMGF). This notebook validates architecture and execution, and reports no predictive performance.

## Controlled sequential design

A0: all five mechanisms off. A1: MSS enabled. A2: morphology supervision added. A3: BAA added. A4: contextual global path added alongside the local path, with element-wise mean fusion. A5: frozen full model with AMGF. Encoder widths 48/80/128/192, depths 2/2/3/2, expansion 2, 192-dimensional classifier input, dropout 0.30 and MRI resolution remain fixed. Standard MFE channel gates remain present in every variant; this is not an ablation of those blocks.

## Imports and reproducibility

In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import os, sys
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
ROOT = Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0,str(ROOT))
import pandas as pd
from IPython.display import display
from src.models.ablation import VARIANTS
from src.utils.ablation_setup import prepare, validate
pd.set_option('display.max_columns', 16)
pd.set_option('display.precision', 5)

## Frozen experimental protocol

Read and verify immutable split hashes, preprocessing sources, the Phase 5 architecture fingerprint and all five existing main best checkpoints. Only training images are accessed during diagnostics. Phase 7 and Phase 10 results are protected and never used to tune a variant. The original full model, loss and data implementation are not edited.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
context = prepare(ROOT)
display(pd.DataFrame(VARIANTS).T)
print('A5 fingerprint:', context['fingerprint']['fingerprint_sha256'])
print('All five frozen A5 best checkpoints loaded strictly for compatibility; no test inference.')

## Configuration rules and model factory

`src/models/ablation.py` reuses the frozen model constructor and blocks. BAA without morphology supervision and AMGF without a dual path raise errors. Non-Boolean flags and structural changes are rejected. A5 is the original `MorphoNetBT` class itself, not a modified wrapper. A0–A4 share one controlled implementation.

Public inference is `logits = model(image)` with MRI `[B,1,256,256]` and logits `[B,3]`. Training may request `return_aux=True`; disabled morphology heads are not instantiated.

## A0 — Core backbone

A conventional 3×3 convolution (1→32, stride 2, padding 1, no bias), BatchNorm and SiLU replaces MSS. The same four MFE encoder stages are retained. Between Stage 3 and Stage 4, the existing local path supplies depthwise 3×3 convolution, BatchNorm/SiLU, then pointwise 1×1 convolution and BatchNorm/SiLU, preserving `[B,128,16,16]`. No boundary attention, morphology heads, global branch or fusion gate is present. The frozen pooling, dropout and 192→3 classifier remain.

## A1 — Add MSS

Restore the original three-branch multi-scale stem and projection to `[B,32,128,128]`. Everything after the stem is identical in structure to A0. Classification-only CE loss remains.

## A2 — Add privileged morphology supervision

Restore the original Stage-2 morphology head and region, boundary and distance predictions for training. BAA remains absent. MRI-only inference entirely skips the morphology head because its outputs are not needed for classification at this step. Head parameters are reported in total/trainable counts even though their operations are absent from inference MACs.

## A3 — Add BAA

Enable the existing attention inside the same local path, between its depthwise and pointwise convolutions. Guidance comes exclusively from the Stage-2 MRI-predicted boundary logits, never GT boundaries. The shared morphology bottleneck and boundary predictor now participate in inference; region and distance outputs remain auxiliary-only.

## A4 — Add Global/Local dual path

Add the original contextual branch: depthwise 5×5 dilation 2 followed by pointwise convolution, both with BatchNorm/SiLU. Fuse it with the unchanged local branch using `0.5 * (global + local)`. This parameter-free, non-adaptive mean introduces no new attention mechanism. The so-called global branch provides dilated context, not all-to-all global attention.

## A5 — Frozen full MorphoNet-BT

Use the exact original class and frozen configuration, including AMGF. Require 579,599 parameters and exact Phase 5 fingerprint equality. Verify strict state-dictionary compatibility with five frozen best checkpoints and exact synthetic-output equivalence with the original model. Existing full-model checkpoints/results will serve as A5 in Phase 12; no experimental A5 retraining is planned.

## Loss and training-data policy

A0/A1 use CE only, and their loss function receives labels without morphology targets. A2–A5 call the unchanged Phase 6 `MorphoNetLoss`:

$L=L_{CE}+0.3[(BCE+Dice)_{region}+(BCE+Dice)_{boundary}+0.5\,SmoothL1_{distance}]$.

Frozen augmentation is identical across variants. As previously approved, training masks may be used internally to reject tumor-clipping transforms. They never enter a model forward call, and A0/A1 loss ignores all morphology targets. No auxiliary weight tuning is allowed.

## Validation execution

Use synthetic MRI for shape/inference checks and the first real Fold 1 training batch for loss/backward validation. All trainable tensors must receive finite, nonzero gradients. Independent layer hooks and THOP must agree on Conv/Linear MACs. Finally, a fresh temporary model per variant performs one successful AMP AdamW step at batch 32. If AMP overflows, only normal GradScaler adaptation is allowed, with bounded retries recorded; no hyperparameters change. Temporary models are discarded and no experimental checkpoints are written.

Executing this cell reruns diagnostics only, never a full training loop.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
tables, audit = validate(context)

## Shape audit

Stem and encoder dimensions are asserted identical; auxiliary predictions are full-resolution single-channel maps.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['ablation_shape_audit'])

## Parameter counts and ablation summary

Counts include every trainable auxiliary head. Removed components are not replaced with dummy parameters.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['ablation_architecture_summary'])

## Computational complexity

Input is [1,1,256,256]. Primary inference excludes unused training-only heads. Auxiliary forward counts are separate and do not include backward/optimizer work. One MAC equals two FLOPs; BN, activation, interpolation, pooling and element-wise arithmetic are excluded consistently with Phase 5.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['ablation_complexity'])

## Real-batch loss validation

Untrained-model diagnostic losses only, not performance estimates. Empty auxiliary cells mean not applicable for CE-only variants, not computed zero losses.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['ablation_real_batch_losses'])

## Batch-32 AdamW memory preflight

Peak allocated/reserved GPU memory is hardware-specific, recorded for disposable diagnostic models. Each row requires exactly one successful update.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['ablation_memory_preflight'])

## Forward, backward and gradient audit

Full per-parameter evidence is saved as CSV. The compact table below shows the number of checked tensors and whether every tensor has present, finite, nonzero gradients in both FP32 and AMP diagnostics.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(tables['ablation_gradient_audit'].groupby(['Variant','Test']).agg(Tensors=('Parameter','size'),AllPresent=('Present','all'),AllFinite=('Finite','all'),AllNonzero=('Nonzero','all')))

## Phase 12 configuration prepared, training disabled

`configs/ablation.yaml` specifies A0–A4 × five frozen folds = 25 future runs. A5 reuses existing results. Shared settings: 100 epochs, batch 32, AdamW LR 0.001, weight decay 0.0001, cosine T_max 100 and eta_min 0, validation macro-F1 early stopping with patience 15, AMP and seed 42. No per-variant tuning, morphology sub-ablation or lambda sensitivity is introduced. Explicit approval is required before Phase 12.

## Reproducibility metadata and scientific audit

Metadata records source hashes, software/GPU versions, split hashes, frozen preprocessing, checkpoint reuse paths and diagnostic batch identities. Protected artifacts are checked again after all diagnostics.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
display(pd.Series(audit,name="Audit result"))

## Scientific limitations and conclusions

Sequential differences are conditional on the preceding components and depend on their order. They cannot by themselves estimate independent effects or interactions. Mechanism removal changes capacity and computation; report those changes rather than inserting unused parameters. A3→A4 adds both the contextual branch and mean fusion; A4→A5 isolates replacing that mean with AMGF. Shared topology does not imply identical random initialization of every tensor across variants. Future work should use the frozen common seed and protocol without test-driven adjustment.

Architecture preparation is complete only if every audit above passes. Diagnostic losses are not evidence of predictive benefit. No full ablation training, test inference, morphology sub-ablation, lambda sweep or significance test is performed. Await separate Phase 12 approval.